# 4 — Production Pipeline: Color Extraction for All Products

Applies the full two-stage extraction pipeline to all ~9k product images:

1. **Stage 1 — Classify** each image with the ResNet-18 classifier → `swatch`, `bullet_lipstick`, `liquid_lipstick`, `other`
2. **Stage 2 — Extract color** using the best strategy per type:
   - `swatch` → k-means peak color (Model A — no segmentation needed, more efficient)
   - `bullet_lipstick`, `liquid_lipstick` → U-Net segmentation + median LAB
   - `other` → U-Net segmentation (other model) + dominant cluster LAB
3. **Post-process** → LAB → hex, color circle HTML, color grouping for app filtering
4. **Save** `data/processed/products_pipeline.csv`

# Libraries

In [1]:
import os
import sys
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.vq
from PIL import Image
from tqdm.auto import tqdm
from skimage import color as skcolor
from skimage.color import deltaE_ciede2000
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.cluster import KMeans
warnings.filterwarnings('ignore')

# Setup

In [2]:
BASE       = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
IMG_DIR    = os.path.join(BASE, 'data', 'img', 'original_clean')
ANN_DIR    = os.path.join(BASE, 'data', 'annotations')
PROC_DIR   = os.path.join(BASE, 'data', 'processed')
MODELS_DIR = os.path.join(BASE, 'models')

RANDOM_SEED = 42
CLASSES     = ['bullet_lipstick', 'liquid_lipstick', 'other', 'swatch']
SEG_CLASSES = {'bullet_lipstick', 'liquid_lipstick'}
DEVICE      = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

# Build a stem→path index so we can match regardless of extension
_clean_index = {}
for fname in os.listdir(IMG_DIR):
    stem = re.sub(r'\.[^.]+$', '', fname)
    _clean_index[stem] = os.path.join(IMG_DIR, fname)

def find_image(img_name):
    """Resolve an img_name to a full path, tolerating extension mismatches."""
    direct = os.path.join(IMG_DIR, img_name)
    if os.path.exists(direct):
        return direct
    stem = re.sub(r'\.[^.]+$', '', img_name)
    return _clean_index.get(stem)

print(f'device : {DEVICE}')
print(f'images in original_clean: {len(os.listdir(IMG_DIR))}')

device : mps
images in original_clean: 9167


# Product Data

In [ ]:
# Scan all images in original_clean directly
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
img_fnames = sorted([f for f in os.listdir(IMG_DIR) if os.path.splitext(f)[1].lower() in IMG_EXTS])

img_df = pd.DataFrame({
    'img_name': img_fnames,
    'img_path': [os.path.join(IMG_DIR, f) for f in img_fnames],
})

print(f'{len(img_df)} images found in {IMG_DIR}')

# Load Models

In [4]:
clf_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
seg_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Classifier
clf = models.resnet18(weights=None)
clf.fc = nn.Linear(clf.fc.in_features, len(CLASSES))
clf.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'resnet18_classifier_al.pth'), map_location=DEVICE)['model_state_dict'])
clf = clf.to(DEVICE).eval()

# Segmenters
def load_seg(ckpt):
    m = smp.Unet(encoder_name='resnet18', encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE)['model_state_dict'])
    return m.eval()

seg      = load_seg(os.path.join(MODELS_DIR, 'unet_segmenter.pth'))
seg_other = load_seg(os.path.join(MODELS_DIR, 'unet_segmenter_other.pth'))

print('classifier + 2 segmenters loaded')

classifier + 2 segmenters loaded


# Stage 1 — Classify All Images

Batch inference through ResNet-18 to predict image type for every product.

In [ ]:
class ImgDataset(Dataset):
    def __init__(self, paths, tf):
        self.paths = paths
        self.tf    = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert('RGB')), i
        except Exception:
            return torch.zeros(3, 224, 224), i

paths  = img_df['img_path'].tolist()
loader = DataLoader(ImgDataset(paths, clf_tf), batch_size=64, num_workers=0)
idx2pred = {}

print(f'Classifying {len(paths)} images ({len(loader)} batches of 64)...')
with torch.no_grad():
    pbar = tqdm(total=len(paths), desc='images classified')
    for batch, indices in loader:
        probs = torch.softmax(clf(batch.to(DEVICE)), dim=1).cpu().numpy()
        for prob, idx in zip(probs, indices.numpy()):
            idx2pred[int(idx)] = (CLASSES[prob.argmax()], float(prob.max()))
        pbar.update(len(indices))
    pbar.close()

img_df['pred_label']      = [idx2pred[i][0] for i in range(len(img_df))]
img_df['pred_confidence'] = [idx2pred[i][1] for i in range(len(img_df))]

print('\nPredicted label distribution:')
print(img_df['pred_label'].value_counts().to_string())

# Stage 2 — Extract Color per Image

Routing:
- `swatch` → k-means peak color (no segmentation — faster and more accurate for swatches)
- `bullet_lipstick` / `liquid_lipstick` → U-Net segmentation + median LAB
- `other` → U-Net (other) segmentation + dominant cluster LAB

In [ ]:
CIELAB_WHITE = np.array([100, 0, 0])
CIELAB_BLACK = np.array([0,   0, 0])
BW_THRESHOLD = 10

def predict_mask(model, img_path, threshold=0.5):
    img = Image.open(img_path).convert('RGB')
    orig_w, orig_h = img.size
    x = seg_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).squeeze().cpu().numpy()
    mask = (prob > threshold).astype(np.uint8)
    return np.array(Image.fromarray(mask * 255).resize((orig_w, orig_h), Image.NEAREST)) // 255

def extract_swatch_peak(img_path, k=3):
    """K-means peak color — best strategy for swatch images."""
    im = Image.open(img_path).convert('RGB').resize((150, 150))
    ar = np.asarray(im).reshape(-1, 3).astype(float)
    codes, _ = scipy.cluster.vq.kmeans(ar, k)
    labels, _ = scipy.cluster.vq.vq(ar, codes)
    counts    = np.bincount(labels)
    codes_lab = np.array([skcolor.rgb2lab([[c / 255.0]])[0][0] for c in codes])
    mask = np.array([
        deltaE_ciede2000(lab, CIELAB_WHITE) >= BW_THRESHOLD and
        deltaE_ciede2000(lab, CIELAB_BLACK) >= BW_THRESHOLD
        for lab in codes_lab
    ])
    if not mask.any():
        mask = np.ones(len(codes), dtype=bool)
    filtered_lab    = codes_lab[mask]
    filtered_counts = counts[mask]
    return filtered_lab[np.argmax(filtered_counts)]

def extract_masked_median(img_rgb, mask_arr):
    """Median LAB of pixels inside the segmentation mask."""
    lab     = skcolor.rgb2lab(img_rgb / 255.0)
    pixels  = lab[mask_arr == 1]
    return np.median(pixels, axis=0) if len(pixels) > 0 else np.full(3, np.nan)

def extract_dominant_cluster(img_rgb, mask_arr, k=3):
    """Dominant k-means cluster of masked pixels — best for 'other' (transparent containers)."""
    lab    = skcolor.rgb2lab(img_rgb / 255.0)
    pixels = lab[mask_arr == 1]
    if len(pixels) == 0:
        return np.full(3, np.nan)
    if len(pixels) < k:
        return np.median(pixels, axis=0)
    km  = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init='auto')
    lbs = km.fit_predict(pixels)
    return km.cluster_centers_[np.bincount(lbs).argmax()]

print('extraction helpers defined')

In [ ]:
L_vals, a_vals, b_vals = [], [], []

for _, row in tqdm(img_df.iterrows(), total=len(img_df), desc='extracting color'):
    img_path = row['img_path']
    label    = row['pred_label']
    try:
        if label == 'swatch':
            lab = extract_swatch_peak(img_path)
        elif label in SEG_CLASSES:
            img  = np.array(Image.open(img_path).convert('RGB'))
            mask = predict_mask(seg, img_path)
            lab  = extract_masked_median(img, mask)
        else:  # other
            img  = np.array(Image.open(img_path).convert('RGB'))
            mask = predict_mask(seg_other, img_path)
            lab  = extract_dominant_cluster(img, mask)
    except Exception:
        lab = np.full(3, np.nan)

    L_vals.append(float(lab[0]))
    a_vals.append(float(lab[1]))
    b_vals.append(float(lab[2]))

img_df['L'] = L_vals
img_df['a'] = a_vals
img_df['b'] = b_vals

print(f'Color extracted: {img_df["L"].notna().sum()} / {len(img_df)}')

# Post-processing

Convert LAB → RGB → hex and generate the HTML color circle used in the app.

In [ ]:
def lab_to_hex(L, a, b):
    rgb = skcolor.lab2rgb(np.array([[[L, a, b]]])).clip(0, 1)[0, 0]
    r, g, bv = (rgb * 255).round().astype(int)
    return f'#{r:02x}{g:02x}{bv:02x}'

def make_circle(hex_color, size=20):
    return (f'<div style="width:{size}px; height:{size}px; '
            f'border-radius:50%; background-color:{hex_color};"></div>')

valid = img_df['L'].notna()
img_df.loc[valid, 'hex_color'] = img_df[valid].apply(
    lambda r: lab_to_hex(r['L'], r['a'], r['b']), axis=1
)
img_df.loc[valid, 'Circle'] = img_df.loc[valid, 'hex_color'].apply(make_circle)

print(f'{valid.sum()} rows with hex color')
img_df[['img_name', 'pred_label', 'L', 'a', 'b', 'hex_color', 'Circle']].head(4)

# Color Grouping

Cluster all extracted hex colors into a small set of representative groups using k-means in LAB space. This produces a `color_group` label and a `group_hex` representative color per group — used by the app for color-based filtering.

The elbow method is used to pick the number of groups.

In [ ]:
color_df = img_df[img_df['L'].notna()].copy()
X = color_df[['L', 'a', 'b']].values

K_RANGE = range(5, 31)
inertias = [KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10).fit(X).inertia_ for k in tqdm(K_RANGE, desc='elbow')]

d2 = np.diff(np.diff(inertias))
k_opt = list(K_RANGE)[np.argmax(d2) + 1]

plt.figure(figsize=(8, 3))
plt.plot(list(K_RANGE), inertias, marker='o', markersize=4)
plt.axvline(x=k_opt, color='red', linestyle='--', label=f'k={k_opt}')
plt.title('Elbow — number of color groups')
plt.xlabel('k'); plt.ylabel('inertia')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Optimal color groups: k={k_opt}')

In [ ]:
km = KMeans(n_clusters=k_opt, random_state=RANDOM_SEED, n_init=10).fit(X)

color_df = color_df.copy()
color_df['color_group'] = km.labels_

centroids = km.cluster_centers_
group_hex = {i: lab_to_hex(*centroids[i]) for i in range(k_opt)}
color_df['group_hex']    = color_df['color_group'].map(group_hex)
color_df['group_circle'] = color_df['group_hex'].apply(make_circle)

img_df = img_df.merge(
    color_df[['img_name', 'color_group', 'group_hex', 'group_circle']],
    on='img_name', how='left'
)

# Show color palette
fig, axes = plt.subplots(1, k_opt, figsize=(k_opt * 0.8, 1.5))
for ax, i in zip(axes, range(k_opt)):
    rgb = skcolor.lab2rgb(np.array([[[*centroids[i]]]])).clip(0, 1)[0, 0]
    ax.imshow([[rgb]])
    n = (color_df['color_group'] == i).sum()
    ax.set_title(f'{n}', fontsize=7)
    ax.axis('off')
plt.suptitle(f'Color groups (k={k_opt}) — count per group', y=1.05, fontsize=9)
plt.tight_layout()
plt.show()

# Save Output

In [ ]:
# Join with product metadata (brand, product, shade, img_url, etc.)
meta = pd.read_csv(os.path.join(PROC_DIR, 'products_with_images.csv'))

# Normalise to stem so we match regardless of .jpg vs .png extension
img_df['img_stem']  = img_df['img_name'].apply(lambda x: re.sub(r'\.[^.]+$', '', x))
meta['img_stem']    = meta['img_name'].apply(lambda x: re.sub(r'\.[^.]+$', '', x))

meta_cols = ['img_stem', 'id', 'category', 'brand', 'product', 'shade', 'img_url', 'shade_description_original']
out = img_df.merge(meta[[c for c in meta_cols if c in meta.columns]], on='img_stem', how='left')

print(f'{out["brand"].notna().sum()} / {len(out)} images matched to product metadata')

In [ ]:
out_cols = [
    'id', 'category', 'brand', 'product', 'shade', 'shade_description_original',
    'img_url', 'img_name', 'img_path',
    'pred_label', 'pred_confidence',
    'L', 'a', 'b',
    'hex_color', 'Circle',
    'color_group', 'group_hex', 'group_circle',
]
out_final = out[[c for c in out_cols if c in out.columns]].copy()

out_path = os.path.join(PROC_DIR, 'products_pipeline.csv')
out_final.to_csv(out_path, index=False)
print(f'Saved {len(out_final)} rows → {out_path}')
print(f'Columns: {out_final.columns.tolist()}')

# Preview Results

In [ ]:
print('Pipeline summary')
print('='*50)
print(f"Total images processed   : {len(out_final)}")
print(f"Colors extracted         : {out_final['L'].notna().sum()}")
print(f"Colors missing           : {out_final['L'].isna().sum()}")
print(f"Matched to product meta  : {out_final['brand'].notna().sum()}")
print()
print('Predicted label distribution:')
print(out_final['pred_label'].value_counts().to_string())

In [ ]:
# Sample grid: product image + extracted color swatch (4 per predicted category)
n_show = 4

def lab_patch(L, a, b, w=80, h=40):
    rgb = skcolor.lab2rgb(np.array([[[L, a, b]]])).clip(0, 1)
    return np.ones((h, w, 3), dtype=np.float32) * rgb[0, 0]

for label in CLASSES:
    grp = out_final[(out_final['pred_label'] == label) & out_final['L'].notna()].head(n_show).reset_index(drop=True)
    if grp.empty:
        continue
    fig, axes = plt.subplots(2, len(grp), figsize=(len(grp) * 3, 5))
    if len(grp) == 1:
        axes = axes.reshape(2, 1)
    for i, row in grp.iterrows():
        img = np.array(Image.open(row['img_path']).convert('RGB'))
        axes[0, i].imshow(img)
        title = f"{row.get('brand', '')} / {row.get('shade', '')}" if pd.notna(row.get('brand')) else row['img_name'][:30]
        axes[0, i].set_title(title, fontsize=7)
        axes[0, i].axis('off')
        axes[1, i].imshow(lab_patch(row['L'], row['a'], row['b']))
        axes[1, i].set_title(row['hex_color'], fontsize=8)
        axes[1, i].axis('off')
    fig.suptitle(f'{label}', fontsize=10)
    plt.tight_layout()
    plt.show()